In [23]:
import torch
import torch.nn as nn
import torchvision.transforms as transforms
import torch.nn.functional as f
import torch_directml
from PIL import Image
import numpy as np

In [24]:
device = torch_directml.device(0)
print(torch_directml.device_name(0))

AMD Radeon RX 6800S 


In [25]:
# Image processor
def process_image(path):
    img = Image.open(path)
    img = img.resize((256, 256))
    img = torch.FloatTensor(np.array(img)).to(device)
    img = torch.permute(img, (2, 0, 1))
    img = img / 255.0
    normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                     std=[0.229, 0.224, 0.225])
    transformer = transforms.Compose([normalize])
    img = transformer(img)
    return img

In [26]:
#  Model definition
class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 6, 5)
        self.pool = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(6, 12, 5)
        self.fc1 = nn.Linear(12 * 61 * 61, 120)
        self.fc2 = nn.Linear(120, 10)
    def forward(self, x):
        x = self.conv1(x)
        x = f.relu(x)
        x = self.pool(x)
        x = self.conv2(x)
        x = f.relu(x)
        x = self.pool(x)
        x = x.view(-1, 12 * 61 * 61)
        x = self.fc1(x)
        x = f.relu(x)
        x = self.fc2(x)
        return x

In [27]:
# Objects
model = CNN()
model = model.to(device)
optimizer = torch.optim.SGD(model.parameters(),
                            lr=0.001,
                            momentum=0.9)

In [31]:
# Image
image = process_image('./source/images/dog.jpg')
image = image.unsqueeze(0)

In [32]:
output = model(image)
print(output.shape)

torch.Size([1, 10])


In [35]:
print("Model's state dict")
for param_tensor, param_values in model.state_dict().items():
    print(param_tensor, '\t', param_values.size())

Model's state dict
conv1.weight 	 torch.Size([6, 3, 5, 5])
conv1.bias 	 torch.Size([6])
conv2.weight 	 torch.Size([12, 6, 5, 5])
conv2.bias 	 torch.Size([12])
fc1.weight 	 torch.Size([120, 44652])
fc1.bias 	 torch.Size([120])
fc2.weight 	 torch.Size([10, 120])
fc2.bias 	 torch.Size([10])


In [36]:
# Saving
torch.save(model.state_dict(), './source/cnn_model.pth.tar')

Loading model

In [40]:
# Load
new_model = CNN()
new_model.load_state_dict(torch.load('./source/cnn_model.pth.tar'))
new_model.eval()

C:\Users\korol\AppData\Local\Temp\ipykernel_3932\2269319252.py:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  new_model.load_state_dict(torch.load('./source/cnn_model.pth.

CNN(
  (conv1): Conv2d(3, 6, kernel_size=(5, 5), stride=(1, 1))
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv2): Conv2d(6, 12, kernel_size=(5, 5), stride=(1, 1))
  (fc1): Linear(in_features=44652, out_features=120, bias=True)
  (fc2): Linear(in_features=120, out_features=10, bias=True)
)

Saving and loading a checkpoint

In [41]:
checpoint = {
    'epoch' : 1,
    'model_state_dict' : model.state_dict(),
    'optimizer_state_dict' : optimizer.state_dict(),
    'loss' : 0.2
}
torch.save(checpoint, './source/cnn_model_checkpoint.pth.tar')

In [42]:
new_model_checpoint = CNN().to(device)
optimizer = torch.optim.SGD(new_model_checpoint.parameters(),
                            lr=0.001,
                            momentum=0.9)
checkpoint_loaded = torch.load('./source/cnn_model_checkpoint.pth.tar')

C:\Users\korol\AppData\Local\Temp\ipykernel_3932\4073887963.py:5: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint_loaded = torch.load('./source/cnn_model_checkpoint

In [43]:
model.load_state_dict(checkpoint_loaded['model_state_dict'])
optimizer.load_state_dict(checkpoint_loaded['optimizer_state_dict'])
epoch = checkpoint_loaded['epoch']
loss = checkpoint_loaded['loss']
model.train()

CNN(
  (conv1): Conv2d(3, 6, kernel_size=(5, 5), stride=(1, 1))
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv2): Conv2d(6, 12, kernel_size=(5, 5), stride=(1, 1))
  (fc1): Linear(in_features=44652, out_features=120, bias=True)
  (fc2): Linear(in_features=120, out_features=10, bias=True)
)